# Chicago Affordable Housing Data Cleaning

## Purpose

This notebook cleans Chicago's Affordable Rental Housing Developments dataset for later geographic and resource-allocation analysis.

The cleaning process standardizes text fields, converts numeric columns, removes system-generated fields, checks geographic values, documents missing data and saves a reproducible processed dataset.

## Important interpretation

This dataset represents documented City-supported affordable developments. It does not represent currently vacant units, waiting-list availability or housing that is immediately accessible to someone experiencing homelessness.

In [18]:
from pathlib import Path

import numpy as np
import pandas as pd

In [19]:
current_folder = Path.cwd()

if current_folder.name == "notebooks":
    project_root = current_folder.parent
else:
    project_root = current_folder

print("Project root:", project_root)

Project root: /workspaces/chicago-homelessness-resource-allocation/chicago-homelessness-resource-allocation


In [20]:
raw_path = (
    project_root
    / "data"
    / "raw"
    / "affordable_rental_housing_developments.csv"
)

affordable_housing = pd.read_csv(raw_path)

print("Raw dataset dimensions:", affordable_housing.shape)
display(affordable_housing.head())

Raw dataset dimensions: (598, 19)


,community_area,community_area_number,property_type,property_name,address,zip_code,phone_number,management_company,units,x_coordinate,y_coordinate,latitude,longitude,location,:@computed_region_awaf_s7ux,:@computed_region_43wa_7qmu,:@computed_region_vrxf_vc4k,:@computed_region_6mkv_f3dw,:@computed_region_bdys_3d7i
0,Avondale,21,Multifamily,Hairpin Lofts,3414 W. Diversey Ave.,60647,773-292-6360,Leasing & Management Co. Inc.,25.0,1153078.890,1918447.998,41.932073,-87.712872,NaN,NaN,NaN,NaN,NaN,NaN
1,Loop,32,ARO,1000M,1000 S. Michigan Ave.,60605,312-820-1000,Willow Bridge,23.0,1177375.505,1895971.036,41.869878,-87.624269,NaN,NaN,NaN,NaN,NaN,NaN
2,Logan Square,22,ARO,2556 Armtiage LLC,2556 W. Armitage Ave,60647,773-252-0600,North Clybourn Group,1.0,1158751.315,1913231.215,41.917643,-87.692170,"{'latitude': '41.917642826462', 'longitude': '...",24.0,41.0,23.0,22535.0,294.0
3,Douglas,35,Multifamily,South Park Plaza,2600 S. King Dr.,60616,312-674-9210,Woodlawn Comm. Dev. Corp.,134.0,1179206.472,1887158.196,41.845653,-87.617816,"{'latitude': '41.8456529117633', 'longitude': ...",1.0,10.0,1.0,21194.0,191.0
4,Near West Side,28,ARO,The Rosie,1461 S. Blue Island Ave.,60608,872-259-7452,The FLATS,7.0,1168331.384,1892984.019,41.861881,-87.657558,"{'latitude': '41.86188117554516', 'longitude':...",8.0,26.0,29.0,14920.0,96.0


In [21]:
print("Columns in the raw dataset:")

for column in affordable_housing.columns:
    print(column)

Columns in the raw dataset:
community_area
community_area_number
property_type
property_name
address
zip_code
phone_number
management_company
units
x_coordinate
y_coordinate
latitude
longitude
location
:@computed_region_awaf_s7ux
:@computed_region_43wa_7qmu
:@computed_region_vrxf_vc4k
:@computed_region_6mkv_f3dw
:@computed_region_bdys_3d7i


In [22]:
selected_columns = [
    "community_area",
    "community_area_number",
    "property_type",
    "property_name",
    "address",
    "zip_code",
    "phone_number",
    "management_company",
    "units",
    "latitude",
    "longitude",
]

missing_columns = [
    column
    for column in selected_columns
    if column not in affordable_housing.columns
]

assert not missing_columns, f"Required columns are missing: {missing_columns}"

housing_clean = affordable_housing[selected_columns].copy()

print("Selected dataset dimensions:", housing_clean.shape)
display(housing_clean.head())

Selected dataset dimensions: (598, 11)


,community_area,community_area_number,property_type,property_name,address,zip_code,phone_number,management_company,units,latitude,longitude
0,Avondale,21,Multifamily,Hairpin Lofts,3414 W. Diversey Ave.,60647,773-292-6360,Leasing & Management Co. Inc.,25.0,41.932073,-87.712872
1,Loop,32,ARO,1000M,1000 S. Michigan Ave.,60605,312-820-1000,Willow Bridge,23.0,41.869878,-87.624269
2,Logan Square,22,ARO,2556 Armtiage LLC,2556 W. Armitage Ave,60647,773-252-0600,North Clybourn Group,1.0,41.917643,-87.692170
3,Douglas,35,Multifamily,South Park Plaza,2600 S. King Dr.,60616,312-674-9210,Woodlawn Comm. Dev. Corp.,134.0,41.845653,-87.617816
4,Near West Side,28,ARO,The Rosie,1461 S. Blue Island Ave.,60608,872-259-7452,The FLATS,7.0,41.861881,-87.657558


In [23]:
numeric_columns = [
    "community_area_number",
    "units",
    "latitude",
    "longitude",
]

for column in numeric_columns:
    housing_clean[column] = pd.to_numeric(
        housing_clean[column],
        errors="coerce",)

In [24]:
numeric_columns = [
    "community_area_number",
    "units",
    "latitude",
    "longitude",
]

for column in numeric_columns:
    housing_clean[column] = pd.to_numeric(
        housing_clean[column],
        errors="coerce",
    )

housing_clean["community_area_number"] = (
    housing_clean["community_area_number"].astype("Int64")
)

housing_clean["units"] = housing_clean["units"].astype("Int64")

display(housing_clean[numeric_columns].head())

,community_area_number,units,latitude,longitude
0,21,25,41.932073,-87.712872
1,32,23,41.869878,-87.624269
2,22,1,41.917643,-87.692170
3,35,134,41.845653,-87.617816
4,28,7,41.861881,-87.657558


In [25]:
housing_clean["units_missing"] = housing_clean["units"].isna()

print("Rows with missing unit counts:")
display(
    housing_clean.loc[
        housing_clean["units_missing"],
        [
            "property_name",
            "address",
            "community_area",
            "units",
            "units_missing",
        ],
    ]
)

Rows with missing unit counts:


,property_name,address,community_area,units,units_missing
27,Arthur on Aberdeen,210 N. Aberdeen St.,Near West Side,<NA>,True


In [26]:
invalid_community_areas = housing_clean.loc[
    housing_clean["community_area_number"].notna()
    & ~housing_clean["community_area_number"].between(1, 77)
]

invalid_latitudes = housing_clean.loc[
    housing_clean["latitude"].notna()
    & ~housing_clean["latitude"].between(41.6, 42.1)
]

invalid_longitudes = housing_clean.loc[
    housing_clean["longitude"].notna()
    & ~housing_clean["longitude"].between(-88.0, -87.5)
]

print("Invalid community-area numbers:", len(invalid_community_areas))
print("Suspicious latitude values:", len(invalid_latitudes))
print("Suspicious longitude values:", len(invalid_longitudes))

Invalid community-area numbers: 0
Suspicious latitude values: 0
Suspicious longitude values: 0


In [27]:
duplicate_count = housing_clean.duplicated().sum()

missing_summary = (
    housing_clean.isna()
    .sum()
    .rename("missing_count")
    .to_frame()
)

missing_summary["missing_percent"] = (
    housing_clean.isna().mean() * 100
).round(1)

print("Exact duplicate rows:", duplicate_count)
display(missing_summary.sort_values("missing_percent", ascending=False))

Exact duplicate rows: 0


,missing_count,missing_percent
units,1,0.2
community_area,0,0.0
property_type,0,0.0
community_area_number,0,0.0
property_name,0,0.0
address,0,0.0
phone_number,0,0.0
zip_code,0,0.0
management_company,0,0.0
latitude,0,0.0


In [28]:
assert not housing_clean.empty, "The cleaned dataset is empty."
assert housing_clean.columns.is_unique, "Duplicate column names were found."
assert housing_clean["community_area_number"].dropna().between(1, 77).all()
assert housing_clean["units"].dropna().gt(0).all()

print("All cleaning validation checks passed.")

All cleaning validation checks passed.


In [29]:
clean_path = (
    project_root
    / "data"
    / "processed"
    / "affordable_housing_cleaned.csv"
)

housing_clean.to_csv(clean_path, index=False)

saved_clean = pd.read_csv(clean_path)

assert saved_clean.shape == housing_clean.shape

print("Cleaned dataset saved to:", clean_path)
print("Saved dataset dimensions:", saved_clean.shape)
display(saved_clean.head())

Cleaned dataset saved to: /workspaces/chicago-homelessness-resource-allocation/chicago-homelessness-resource-allocation/data/processed/affordable_housing_cleaned.csv
Saved dataset dimensions: (598, 12)


,community_area,community_area_number,property_type,property_name,address,zip_code,phone_number,management_company,units,latitude,longitude,units_missing
0,Avondale,21,Multifamily,Hairpin Lofts,3414 W. Diversey Ave.,60647,773-292-6360,Leasing & Management Co. Inc.,25.0,41.932073,-87.712872,False
1,Loop,32,ARO,1000M,1000 S. Michigan Ave.,60605,312-820-1000,Willow Bridge,23.0,41.869878,-87.624269,False
2,Logan Square,22,ARO,2556 Armtiage LLC,2556 W. Armitage Ave,60647,773-252-0600,North Clybourn Group,1.0,41.917643,-87.692170,False
3,Douglas,35,Multifamily,South Park Plaza,2600 S. King Dr.,60616,312-674-9210,Woodlawn Comm. Dev. Corp.,134.0,41.845653,-87.617816,False
4,Near West Side,28,ARO,The Rosie,1461 S. Blue Island Ave.,60608,872-259-7452,The FLATS,7.0,41.861881,-87.657558,False
